# TalentCLEF 2026 Task A - Query Augmentation Cache Builder

This notebook generates deterministic query augmentations outside the retrieval notebooks.

It uses `task_a_query_augmentation.py` to:
- parse each job description into structured fields
- build reproducible `profile` and `ideal resume` text views
- cache one JSON file per query under `var/cache/query_augmentation/`
- write a compact CSV summary under `var/artifacts/query_augmentation/`

Reproducibility rules:
- no sampling
- no external APIs
- fixed augmentation version string
- cache key derived from raw query text + language + augmentation version


In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "task_a_query_augmentation.py").exists():
    if (PROJECT_ROOT.parent / "task_a_query_augmentation.py").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent
    else:
        raise RuntimeError("Run this notebook from the repository root or the notebooks/ directory.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from task_a_jobbert_dense_en_es import load_task_a_inputs
from task_a_query_augmentation import (
    AUGMENTATION_GENERATOR,
    AUGMENTATION_VERSION,
    augment_topics,
    summarize_augmentation,
)

PROJECT_ROOT


In [ ]:
CACHE_ROOT = PROJECT_ROOT / "var" / "cache" / "query_augmentation"
SUMMARY_ROOT = PROJECT_ROOT / "var" / "artifacts" / "query_augmentation"

LANG_CONFIGS = {
    "en": {
        "development_dir": PROJECT_ROOT / "data" / "development" / "en",
        "test_dir": PROJECT_ROOT / "data" / "test" / "en",
    },
    "es": {
        "development_dir": PROJECT_ROOT / "data" / "development" / "es",
        "test_dir": PROJECT_ROOT / "data" / "test" / "es",
    },
}

SPLIT_NAME = "development"  # or "test"
OVERWRITE_CACHE = False

print("augmentation version:", AUGMENTATION_VERSION)
print("augmentation generator:", AUGMENTATION_GENERATOR)
print("split:", SPLIT_NAME)


In [ ]:
all_summaries = []

for lang, cfg in LANG_CONFIGS.items():
    _, topics = load_task_a_inputs(cfg[f"{SPLIT_NAME}_dir"])
    topics_aug = augment_topics(
        topics=topics,
        lang=lang,
        split_name=SPLIT_NAME,
        cache_root=CACHE_ROOT,
        overwrite=OVERWRITE_CACHE,
    )
    summary = summarize_augmentation(topics_aug)
    out_path = SUMMARY_ROOT / SPLIT_NAME / f"{lang}_query_augmentation_summary.csv"
    out_path.parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(out_path, index=False)
    print(f"saved {lang} summary:", out_path)
    all_summaries.append(summary.assign(lang=lang))

pd.concat(all_summaries, ignore_index=True)


In [ ]:
preview_cols = [
    "lang",
    "qid",
    "title",
    "seniority",
    "skills",
    "experience_terms",
    "compact_rewrite_text",
    "profile_text",
    "ideal_resume_text",
]
pd.concat(all_summaries, ignore_index=True)[preview_cols].head(10)
